# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all data fields and record sets by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, their IDs, fields, and brief details.

Let's query the dataset's record sets and show the fields for each, referencing all by their `@id`.

In [ ]:
# Discover available record sets and their fields with @id references
from pprint import pprint

record_sets = []
field_overview = {}
if hasattr(metadata, 'record_sets'):
    record_sets = list(metadata.record_sets.keys())
    for rs_id, rs in metadata.record_sets.items():
        print(f"Record set: {rs_id}\n  Name: {getattr(rs, 'name', None)}")
        field_overview[rs_id] = []
        if hasattr(rs, 'fields'):
            for field_id, field in rs.fields.items():
                print(f"    Field: {field_id} | Name: {getattr(field, 'name', None)} | Type: {getattr(field, 'data_type', None)}")
                field_overview[rs_id].append((field_id, getattr(field, 'name', None), getattr(field, 'data_type', None)))
        print()
else:
    # fallback: try to use dataset.available_record_sets
    record_sets = list(dataset.available_record_sets())
    print("Available record sets:", record_sets)
    for rs_id in record_sets:
        print(f" - {rs_id}")
        field_overview[rs_id] = []
        rs_schema = dataset._get_recordset_metadata(rs_id)
        if rs_schema and 'fields' in rs_schema:
            for field in rs_schema['fields']:
                print(f"    Field: {field.get('@id', '(missing)')} | Name: {field.get('name', None)} | Type: {field.get('dataType', None)}")
                field_overview[rs_id].append((field.get('@id'), field.get('name'), field.get('dataType')))
        print()
    
print("Summary field overview:")
pprint(field_overview)

## 3. Data Extraction
Load data from all record sets into DataFrames. You can select any of these DataFrames for further analysis, referencing record set and field `@id`s only.

In [ ]:
# Extract all record sets into pandas DataFrames (references by @id)
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading: {record_set_id}")
    records_gen = dataset.records(record_set=record_set_id)
    records = list(records_gen)
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Fields: {dataframes[record_set_id].columns.tolist()}")
        print(f"  First few rows:")
        display(dataframes[record_set_id].head())
    else:
        print(f"  No data for record set: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps to a record set, referencing field and record set `@id`s. We'll select a record set and numeric field for demonstration.

> **Note:** Replace `<selected_record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with values observed in your data overview above for meaningful analysis.

In [ ]:
# Example: Let's pick the first available record set and a numeric field
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"Using record set: {selected_record_set_id}")
    # Guess a numeric field by dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_fields:
        # Try converting columns that look like numbers
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except (ValueError, TypeError):
                continue
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Default to first numeric field
        print(f"Numeric field chosen for analysis: {numeric_field_id}")
        # Set a threshold (e.g., at the median) for demonstration
        threshold = df[numeric_field_id].median()

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Try grouping by a likely categorical field
        candidate_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field_id = None
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (mean values):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields available in record set.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize a numeric field's distribution and a relationship with the group field if available.
Visualizations help spot trends or anomalies in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load dataset metadata and records using the Croissant schema and `mlcroissant`
- List available record sets, fields, and reference them by `@id`
- Load the records into DataFrames for analysis
- Perform basic EDA including filtering, normalization, and grouping
- Visualize distributions and grouped field relationships

You can use this approach on any Croissant-JSON schema compatible dataset, always referencing entities by their `@id` for portability and consistency.